# Calculating the Fisher forecasts

This notebook reproduces every Fisher forecast in Cheslog et al. (2026). It
computes the numerical derivatives of the CMB and BAO theory, builds the
twenty Fisher matrices from them, and reports the forecasted parameter
errors that go into the figures and tables.

It follows the `example_calculate_fisher_matrices.ipynb` notebook in
[hdfisher](https://github.com/CMB-HD/hdfisher), which is the place to start
if you have not used the `Fisher` class before. What we add here are the
nine configurations this paper needs.

Most of the forecasts in the paper are made with CLASS. That includes
Table IV, Figures 2 and 4, and the extended models. CAMB is used where
CLASS cannot be, or where the two codes disagree. Those cases are the
binned $\mathcal{P}(k)$ forecasts, which go through hdPk and CAMB, the
$w_0 w_a$CDM forecasts, where we found CAMB and CLASS differ by up to about
45%, and the baryonic feedback forecast of Table V. A CAMB nine-parameter
forecast is also kept for the comparison with CLASS in Appendix B and for
the accuracy study of Appendix A.

The derivatives are the expensive part. They can be run from this notebook,
and the section on the nine configurations shows how, but one configuration
takes hours at our accuracy settings and there are nine. In practice
`run_hdInitPk_forecasts.py` under MPI is the way to run them. Everything
after that takes seconds and is done here.

## Setup

The step-size and fiducial-parameter files ship with the package, so these
paths work wherever `hdinitpk` is installed. The derivatives are written to
`hdinitpk/data/user_generated_data`, since they are far too large to ship
with the package. The CMB-HD mock data, and the BBN table that CLASS
reads, come from hdMockData.

In [1]:
import os

import numpy as np
import pandas as pd
from IPython.display import display

from hdfisher import fisher
from hd_mock_data import hd_data
from hdinitpk import hdinitPkfisher
import hdinitpk

steps_dir = hdinitpk.data_path('fisher_steps')
params_dir = hdinitpk.data_path('fisher_fid_params')
saved_matrix_dir = hdinitpk.data_path('fisher_matrices')

# Output goes under `hdinitpk/data/user_generated_data`, the same place
# `run_hdInitPk_forecasts.py` writes, so the two are interchangeable.
deriv_dir = hdinitpk.user_data_path('fisher_derivs')
matrix_out_dir = hdinitpk.user_data_path('fisher_matrices')


def steps(name):
    """A step-size file from the package."""
    return os.path.join(steps_dir, name)


def params(name):
    """A fiducial-parameter file from the package."""
    return os.path.join(params_dir, name)


--------------------------------------------------------------------------
but there are no active ports detected (or Open MPI was unable to use
them).  This is most certainly not what you wanted.  Check your
cables, subnet manager configuration, etc.  The openib BTL will be
ignored for this job.

  Local host: login1
--------------------------------------------------------------------------


The settings below are shared by every configuration. We marginalize over
$H_0$ rather than $\theta$ throughout.

The CMB-HD mock data (the fiducial spectra, the noise curves, the binning,
the lensing reconstruction noise, and the covariance matrices) all come
from [hdMockData](https://github.com/CMB-HD/hdMockData). So does the
maximum multipole the theory is calculated to, which is 24,000 for every
configuration, SO-like included. We pin the version rather than leaving it
at hdfisher's default of `'latest'`, so that a new release cannot quietly
move the forecasts. v1.2 is what the paper used.

The $\tau$ prior of 0.005 is applied to every matrix. Where baryonic
feedback is varied we also add a prior on $\log_{10}(T_\mathrm{AGN})$ of
0.06% of its fiducial value of 7.8.

In [2]:
use_H0 = True
hd_data_version = 'v1.2'
hd_datalib = hd_data.HDMockData(version=hd_data_version)

tau_prior = {'tau': 0.005}
feedback_prior = {'tau': 0.005, 'HMCode_logT_AGN': 0.0006 * 7.8}

# k bin edges [Mpc^-1] of the 11-bin binned-P(k) scheme, from the binning
# file (Table I)
lower, upper = np.loadtxt(hdinitpk.data_path('binning', 'hd_pk_wavenumbers_11bins.txt'),
                          usecols=(1, 2), unpack=True)
pk_bin_edges = list(lower) + [upper[-1]]

# the fraction by which the power in a k bin is changed when its amplitude
# is varied; this has to match the 5% steps in the binned-P(k) step files
pk_frac_step = 0.05


In [3]:
nine_params = ['mnu', 'tau', 'logA', 'H0', 'ombh2', 'omch2', 'ns', 'nnu', 'nrun']
eight_params = ['mnu', 'tau', 'logA', 'H0', 'ombh2', 'omch2', 'ns', 'nnu']
feedback_params = nine_params + ['HMCode_logT_AGN', 'n_ksz', 'A_ksz']
w0wa_params = ['ombh2', 'omch2', 'H0', 'tau', 'logA', 'ns', 'nrun', 'w', 'wa']

binned_pk_params = [f'Pk{i+1}' for i in range(11)] + ['H0', 'tau', 'ombh2', 'omch2']
binned_pk_params_so = [f'Pk{i+1}' for i in range(7)] + ['H0', 'tau', 'ombh2', 'omch2']
binned_pk_params_feedback = (binned_pk_params
                             + ['HMCode_logT_AGN', 'n_ksz', 'A_ksz'])

ext_model_params = {
    'lcdm_nrun':          ['ombh2', 'omch2', 'H0', 'tau', 'logA', 'ns', 'nrun'],
    'lcdm_nrun_neff':     ['ombh2', 'omch2', 'H0', 'tau', 'logA', 'ns', 'nrun', 'nnu'],
    'lcdm_nrun_neff_mnu': ['ombh2', 'omch2', 'H0', 'tau', 'logA', 'ns', 'nrun',
                           'nnu', 'mnu'],
}

# The most bins any forecast here uses, in numeric rather than alphabetical
# order, which is what `Pk10` would otherwise sort into.
pk_bin_columns = [f'Pk{i+1}' for i in range(11)]


def split_bins(table):
    """Split a table of parameters into its cosmological and P(k)-bin
    halves.

    Only four of the twenty matrices carry bins, and they carry up to
    eleven each, so a single table is wide enough that pandas hides the
    middle of it. Each half drops the rows and columns that are empty in
    it, so the bin half holds only the binned forecasts.
    """
    cosmo = [c for c in table.columns if c not in pk_bin_columns]
    bins = [c for c in pk_bin_columns if c in table.columns]
    trim = lambda t: t.dropna(axis=0, how='all').dropna(axis=1, how='all')
    return trim(table[cosmo]), trim(table[bins])

The CLASS parameter files use CLASS's own parameter names, so parameter
lists and priors have to be translated on the way into a CLASS Fisher
library, and the errors have to be translated back on the way out. Without
that second step the tables below would carry two naming conventions at
once, and every CLASS row would be blank in the CAMB columns and the other
way around. The saved matrices themselves keep CLASS names, and the
plotting notebook translates them at load time.

In [4]:
camb_to_class = {
    'ombh2': 'omega_b',   'omch2': 'omega_cdm',  'theta': 'theta_s_100',
    'tau': 'tau_reio',    'logA': 'ln_A_s_1e10', 'As': 'A_s',
    'ns': 'n_s',          'H0': 'H0',            'nnu': 'N_ur',
    'mnu': 'sum_m_ncdm',  'nrun': 'alpha_s',     'omk': 'Omega_k',
    'w': 'w0_fld',        'wa': 'wa_fld',
    'HMCode_logT_AGN': 'log10T_heat_hmcode',
}
class_to_camb = {v: k for k, v in camb_to_class.items()}


def to_class(p):
    """CAMB parameter names -> CLASS. Takes a list, or a dict of priors."""
    if isinstance(p, dict):
        return {camb_to_class.get(k, k): v for k, v in p.items()}
    return [camb_to_class.get(k, k) for k in p]


def from_class_errors(errors):
    """`get_fisher_errors` output -> CAMB names.

    Safe to apply to a CAMB result as well, since no CAMB parameter name is
    also a CLASS one. Only the names change, never the values.
    """
    return {class_to_camb.get(p, p): v for p, v in errors.items()}

## Fisher Configuration

`Fisher` takes one required argument, the directory it reads and writes.
Constructing it creates `fiducial_params.yaml` and `step_sizes.yaml` in
that directory, recording what the run was configured with, along with
`theory` and `derivs` subdirectories.

Every configuration here is an `hdinitpk.hdinitPkfisher.Fisher`. It
extends hdfisher's `Fisher` with the binned $\mathcal{P}(k)$ and the kSZ
template, and calculates the theory to the CMB-HD maximum multipole from
hdMockData for every experiment, which is what the paper did for the
SO-like forecasts too.

Here is the nine-parameter CMB-HD CLASS forecast, which is the one behind
most of the paper.

In [10]:
class_9param = hdinitPkfisher.Fisher(
    os.path.join(deriv_dir, 'class_9param'),
    overwrite=False, use_H0=use_H0, use_class_or_camb='class',
    hd_data_version=hd_data_version,
    fisher_steps_file=steps('class_fiducial_step_sizes.yaml'),
    param_file=params('class_fiducial_params.yaml'))


/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hdfisher/fisher.py:884: UserWarning: `use_class=True`: delensed spectra are not calculated with CLASS. No delensed derivatives will be calculated, and a Fisher matrix cannot be calculated from delensed spectra. Use `cmb_type='lensed'`, or set `use_class=False` to use CAMB.
  warnings.warn(msg)


One thing to be careful about. `Fisher.__init__` creates an empty `derivs`
directory when it does not find one, rather than complaining. If it is
pointed somewhere wrong it will not error. It silently recomputes the
fiducial theory instead, which at these accuracy settings runs for hours
in the notebook. When run in a script with mpi, it should take significantly less time (about 30 minutes per set of derivatives). It is worth checking what is there
before building anything.

In [11]:
def deriv_count(fisher_dir):
    d = os.path.join(fisher_dir, 'derivs')
    if not os.path.isdir(d):
        return 0
    return len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))])

print(f'{deriv_count(class_9param.fisher_dir)} derivative files')

27 derivative files


`calculate_fisher_derivs()` steps every varied parameter up and down and
writes the derivatives. Uncomment it to run it here. It works in the
notebook, it is only slow.

The CMB-HD delensing needs the lensing reconstruction noise, which hdfisher
loads from hdMockData at the `hd_data_version` set above. It is worth
checking that it found the version you meant.

In [12]:
# What the delensing will use. `lensing_noise_spectrum` also takes
# `pol_only_lensing=True` for the EE/EB-only reconstruction, which goes with
# the polarization-only covariance matrix used further down.
print(hd_datalib.lensing_noise_fname())
print(hd_datalib.block_covmat_fname('lensed'))
print(hd_datalib.block_covmat_fname('lensed', pol_only_lensing=True))

# class_9param.calculate_fisher_derivs()

/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hd_mock_data/data/noise/hd_lmin30lmax20100Lmax20100_nlkk_v1.2.txt
/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hd_mock_data/data/covariance_matrices/hd_fsky0pt591_lmin30lmax20100_binned_lensed_cov_v1.2.txt
/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hd_mock_data/data/covariance_matrices/hd_fsky0pt591_lmin30lmax20100_binned_lensed_MVpol_cov_v1.2.txt


With the derivatives on disk, `get_fisher_errors` returns the marginalized
$1\sigma$ errors directly. It takes the same arguments as `get_fisher`,
which are the spectra to use, the parameters to include, the priors, and
whether to add the DESI BAO Fisher. Since this is a CLASS library, the
parameter list and the prior go through `to_class`, and the errors come
back with CLASS names.

In [13]:
errs = class_9param.get_fisher_errors(
    'lensed', params=to_class(nine_params), priors=to_class(tau_prior),
    use_H0=use_H0, with_desi=True)

pd.Series(errs, name='CMB-HD + DESI, 9 parameters (CLASS)').to_frame().T

,H0,Neff,alpha_s,ln_A_s_1e10,n_s,omega_b,omega_cdm,sum_m_ncdm,tau_reio
"CMB-HD + DESI, 9 parameters (CLASS)",0.31075,0.015312,0.001231,0.008522,0.001935,0.000026,0.000392,0.028138,0.004562


## The nine configurations

Three things distinguish them. The Boltzmann code, the experiment, and
whether the primordial spectrum is a power law or binned. The binned
$\mathcal{P}(k)$ and the kSZ template are the `binned_pk`, `bin_edges`,
`ksz` and `pk_frac_step` options of `hdinitpk.hdinitPkfisher.Fisher`.

The two CLASS configurations, `class` for CMB-HD and `so` for SO-like, are
the ones behind Table IV and Figures 2 and 4. They take the CLASS step-size
and fiducial-parameter files, so the parameters they vary carry CLASS names
(`omega_b`, `tau_reio`, `ln_A_s_1e10` and so on), and so do the Fisher
matrices built from them. The `to_class` helper above translates a
parameter list or a prior dictionary on the way in.

The other seven run CAMB. The `camb` configuration is the CMB-HD
nine-parameter forecast used for the comparison with CLASS in Appendix B
and for the accuracy study of Appendix A. The two `w0wa` configurations use
CAMB because we found CAMB and CLASS disagree by up to about 45% for that
model. The `camb_feedback_ksz` configuration is the delensed forecast of
Table V. The three binned $\mathcal{P}(k)$ configurations use CAMB because
the binned spectrum goes through hdPk, which is CAMB only.

Every job is built with the same `hd_data_version`, which `calculate_derivs`
supplies below. That is what selects the mock spectra, the noise curves, the
lensing reconstruction noise used for the delensing, and the covariance
matrix.

In [14]:
jobs = {
    # CMB-HD, CLASS. Behind Table IV and Figures 2 and 4.
    'class': dict(
        dirname='class_9param',
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class_or_camb='class',
            fisher_steps_file=steps('class_fiducial_step_sizes.yaml'),
            param_file=params('class_fiducial_params.yaml'))),

    # SO-like, CLASS. Backs the 9-parameter SO forecast and the three
    # extended models of Table IV.
    'so': dict(
        dirname='so_9param',
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class_or_camb='class', exp='so',
            fisher_steps_file=steps('class_fiducial_step_sizes.yaml'),
            param_file=params('class_fiducial_params.yaml'))),

    # CMB-HD, CAMB. For the comparison with CLASS (Appendix B) and the
    # accuracy study (Appendix A).
    'camb': dict(
        dirname='camb_9param',
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class_or_camb='camb',
            fisher_steps_file=steps('camb_fiducial_step_sizes.yaml'),
            param_file=params('camb_fiducial_params.yaml'))),

    # 12 parameters, the nine plus baryonic feedback and the kSZ template.
    # Table V, with delensed spectra.
    'camb_feedback_ksz': dict(
        dirname='camb_feedback_ksz',
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class_or_camb='camb', ksz=True,
            fisher_steps_file=steps('camb_fiducial_step_sizes_feedback_ksz.yaml'),
            param_file=params('camb_fiducial_params_feedback.yaml'))),

    # w0waCDM + alpha_s, with CAMB. Both need the w0wa step-size file,
    # since the default steps do not vary w and wa.
    'w0wa_hd': dict(
        dirname='hd_w0wa',
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class_or_camb='camb',
            fisher_steps_file=steps('camb_w0wa_fiducial_step_sizes.yaml'),
            param_file=params('camb_w0wa_fiducial_params.yaml'))),

    'w0wa_so': dict(
        dirname='so_w0wa',
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class_or_camb='camb', exp='so',
            fisher_steps_file=steps('camb_w0wa_fiducial_step_sizes.yaml'),
            param_file=params('camb_w0wa_fiducial_params.yaml'))),

    # Binned P(k), with CAMB through hdPk.
    'binned_pk_hd': dict(
        dirname='hd_binned_pk',
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class_or_camb='camb',
            binned_pk=True, bin_edges=pk_bin_edges, pk_frac_step=pk_frac_step,
            fisher_steps_file=steps('binned_pk_steps_5_percent.yaml'),
            param_file=params('hd_binned_pk_fiducial_params.yaml'))),

    'binned_pk_so': dict(
        dirname='so_binned_pk',
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class_or_camb='camb', exp='so',
            binned_pk=True, bin_edges=pk_bin_edges, pk_frac_step=pk_frac_step,
            fisher_steps_file=steps('binned_pk_steps_5_percent.yaml'),
            param_file=params('hd_binned_pk_fiducial_params.yaml'))),

    'binned_pk_hd_feedback': dict(
        dirname='hd_binned_pk_feedback',
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class_or_camb='camb',
            binned_pk=True, ksz=True, bin_edges=pk_bin_edges,
            pk_frac_step=pk_frac_step,
            fisher_steps_file=steps('binned_pk_steps_5_percent_feedback.yaml'),
            param_file=params('hd_binned_pk_fiducial_params_feedback.yaml'))),
}

pd.DataFrame([
    {'configuration': name,
     'exp': job['kwargs'].get('exp', 'hd'),
     'code': job['kwargs']['use_class_or_camb'].upper(),
     'binned P(k)': job['kwargs'].get('binned_pk', False),
     'kSZ': job['kwargs'].get('ksz', False),
     'derivatives in': job['dirname']}
    for name, job in jobs.items()]).set_index('configuration')


,class,exp,code,binned P(k),kSZ,derivatives in
configuration,,,,,,
class,hdfisher.Fisher,hd,CLASS,False,False,class_9param
so,hdfisher.Fisher,so,CLASS,False,False,so_9param
camb,hdfisher.Fisher,hd,CAMB,False,False,camb_9param
camb_feedback_ksz,hdinitpk.Fisher,hd,CAMB,False,True,camb_feedback_ksz
w0wa_hd,hdfisher.Fisher,hd,CAMB,False,False,hd_w0wa
w0wa_so,hdfisher.Fisher,so,CAMB,False,False,so_w0wa
binned_pk_hd,hdinitpk.Fisher,hd,CAMB,True,False,hd_binned_pk
binned_pk_so,hdinitpk.Fisher,so,CAMB,True,False,so_binned_pk
binned_pk_hd_feedback,hdinitpk.Fisher,hd,CAMB,True,True,hd_binned_pk_feedback


### Running the derivatives

The loop below does all nine. It takes many hours in a single process, so
it is commented out. The same nine configurations are in
`run_hdInitPk_forecasts.py`, which runs them under MPI.

```
mpirun -np 4 python run_hdInitPk_forecasts.py
```

Set `CALCULATE_DERIVS = True` at the top of that file. Either way the output
lands in the same place, and the rest of this notebook picks it up.

Every configuration above is constructed with `overwrite=True`, so running
this discards any derivatives already in `deriv_dir`. That is what you want
for a clean rebuild and destructive otherwise.

In [15]:
def calculate_derivs(name):
    job = jobs[name]
    out = os.path.join(deriv_dir, job['dirname'])
    print(f'{name} -> {out}', flush=True)
    kwargs = dict(job['kwargs'])
    kwargs.setdefault('hd_data_version', hd_data_version)
    fisherlib = hdinitPkfisher.Fisher(out, **kwargs)
    fisherlib.calculate_fisher_derivs()
    return fisherlib

# for name in jobs:
#     calculate_derivs(name)

In [16]:
# what is actually on disk
pd.DataFrame([
    {'configuration': name,
     'derivative files': deriv_count(os.path.join(deriv_dir, job['dirname']))}
    for name, job in jobs.items()]).set_index('configuration')

,derivative files
configuration,
class,27
so,27
camb,36
camb_feedback_ksz,48
w0wa_hd,44
w0wa_so,44
binned_pk_hd,60
binned_pk_so,60
binned_pk_hd_feedback,72


## The twenty Fisher matrices

Several configurations back more than one matrix. The `class` derivatives
give the nine-parameter CMB-HD forecast of Table IV, the three extended
models of the same table, and the eight-parameter matrix used to compare
against CAMB. The `camb` derivatives give the lensed and delensed
nine-parameter CAMB forecasts and the eight-parameter one used in the
accuracy study. The feedback binned $\mathcal{P}(k)$ derivatives are read
twice, once with the minimum-variance lensing reconstruction and once with
the polarization-only one (`pol_only_lensing=True`).

Four of the six SO matrices come from the CLASS `so` configuration, so
their parameter lists and priors go through `to_class` and the saved
matrices carry CLASS names. The other two stay on CAMB. `so_w0wa_lensed`
comes from the `w0wa_so` configuration, and the binned $\mathcal{P}(k)$
forecast cannot use CLASS.

All the forecasts use lensed spectra, apart from the two Table V matrices,
which use delensed spectra as in the paper. hdMockData carries CMB-HD data
only, so the SO covariance matrices come from hdfisher's own `data/covmats`
directory.

In [17]:
matrices = [
    # (name, configuration, extra kwargs when reading, get_fisher_errors args)
    ('hd_camb_lensed_9param', 'camb', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=True,
          params=nine_params)),
    ('hd_camb_delensed_9param', 'camb', {},
     dict(cmb_type='delensed', priors=tau_prior, with_desi=True,
          params=nine_params)),
    ('hd_camb_lensed_8param', 'camb', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=True,
          params=eight_params)),

    ('hd_class_lensed_9param', 'class', {},
     dict(cmb_type='lensed', priors=to_class(tau_prior), with_desi=True,
          params=to_class(nine_params))),
    ('hd_class_lensed_8param', 'class', {},
     dict(cmb_type='lensed', priors=to_class(tau_prior), with_desi=True,
          params=to_class(eight_params))),

    ('so_class_lensed_9param', 'so', {},
     dict(cmb_type='lensed', priors=to_class(tau_prior), with_desi=True,
          params=to_class(nine_params))),

    ('hd_camb_delensed_12param_feedback', 'camb_feedback_ksz', {},
     dict(cmb_type='delensed', priors=feedback_prior, with_desi=True,
          params=feedback_params)),
    ('hd_camb_delensed_9param_from_feedback', 'camb_feedback_ksz', {},
     dict(cmb_type='delensed', priors=tau_prior, with_desi=True,
          params=nine_params)),

    ('hd_w0wa_lensed', 'w0wa_hd', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=True,
          params=w0wa_params)),
    ('so_w0wa_lensed', 'w0wa_so', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=True,
          params=w0wa_params)),

    ('hd_binned_pk_lensed_11bins', 'binned_pk_hd', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=False,
          params=binned_pk_params)),
    ('so_binned_pk_lensed_7bins', 'binned_pk_so', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=False,
          params=binned_pk_params_so)),

    ('hd_binned_pk_feedback_lensed', 'binned_pk_hd_feedback', {},
     dict(cmb_type='lensed', priors=feedback_prior, with_desi=True,
          params=binned_pk_params_feedback)),
    ('hd_binned_pk_feedback_pol_lensed', 'binned_pk_hd_feedback',
     dict(pol_only_lensing=True),
     dict(cmb_type='lensed', priors=feedback_prior, with_desi=True,
          params=binned_pk_params_feedback)),
]

# LCDM + alpha_s, then + N_eff, then + sum m_nu, for CMB-HD and SO
for model, model_params in ext_model_params.items():
    matrices.append(
        (f'hd_ext_lensed_{model}', 'class', {},
         dict(cmb_type='lensed', priors=to_class(tau_prior), with_desi=True,
              params=to_class(model_params))))
    matrices.append(
        (f'so_ext_lensed_{model}', 'so',
         dict(fisher_params=to_class(['ombh2', 'omch2', 'logA', 'ns', 'tau',
                                      'H0', 'nnu', 'mnu', 'nrun'])),
         dict(cmb_type='lensed', priors=to_class(tau_prior), with_desi=True,
              params=to_class(model_params))))

print(f'{len(matrices)} Fisher matrices from {len(jobs)} configurations')

20 Fisher matrices from 9 configurations


Each one is a `Fisher` built on that configuration's derivatives, then a
call to `get_fisher_errors`. Every library is opened with `overwrite=False`
here. The configurations above carry `overwrite=True` for the derivative
run, and reusing them as they are would clear the derivatives we are about
to read.

In [18]:
def fisher_errors(name, config, extra, call):
    """Marginalized 1-sigma errors for one of the matrices above, keyed by
    CAMB parameter names. A CLASS configuration returns CLASS names, so the
    result goes through `from_class_errors` on the way out."""
    job = jobs[config]
    kwargs = dict(job['kwargs'], overwrite=False)
    kwargs.setdefault('hd_data_version', hd_data_version)
    kwargs.update(extra)
    fisherlib = hdinitPkfisher.Fisher(os.path.join(deriv_dir, job['dirname']), **kwargs)
    return from_class_errors(
        fisherlib.get_fisher_errors(use_H0=use_H0, **call))


forecast_errors = {}
for name, config, extra, call in matrices:
    try:
        forecast_errors[name] = fisher_errors(name, config, extra, call)
        print(f'  {name}')
    except Exception as err:
        print(f'  {name}: {type(err).__name__}: {err}')

print(f'\n{len(forecast_errors)} of {len(matrices)} built')

/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hdfisher/fisher.py:956: UserWarning: Removing 'theta' from the list of varied parameters, because `use_H0=True`.
  warnings.warn(msg)


  hd_camb_lensed_9param
  hd_camb_delensed_9param
  hd_camb_lensed_8param


/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hdfisher/fisher.py:884: UserWarning: `use_class=True`: delensed spectra are not calculated with CLASS. No delensed derivatives will be calculated, and a Fisher matrix cannot be calculated from delensed spectra. Use `cmb_type='lensed'`, or set `use_class=False` to use CAMB.
  warnings.warn(msg)


  hd_class_lensed_9param
  hd_class_lensed_8param


/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hdfisher/fisher.py:807: UserWarning: Ignoring the `hd_lmax` and `include_fg` arguments for `exp = 'so'`.
  warnings.warn(msg1)
/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hdfisher/fisher.py:809: UserWarning: NOTE that we only have mock covariance matrices for *delensed* (as opposed to lensed) CMB spectra for `exp = 'so'` (i.e., we can only calculate a Fisher matrix from delensed power spectra in this case).
  warnings.warn(msg2)


  so_class_lensed_9param
  hd_camb_delensed_12param_feedback
  hd_camb_delensed_9param_from_feedback
  hd_w0wa_lensed
  so_w0wa_lensed
  hd_binned_pk_lensed_11bins
  so_binned_pk_lensed_7bins
  hd_binned_pk_feedback_lensed
  hd_binned_pk_feedback_pol_lensed
  hd_ext_lensed_lcdm_nrun
  so_ext_lensed_lcdm_nrun
  hd_ext_lensed_lcdm_nrun_neff
  so_ext_lensed_lcdm_nrun_neff
  hd_ext_lensed_lcdm_nrun_neff_mnu
  so_ext_lensed_lcdm_nrun_neff_mnu

20 of 20 built


These are the forecasted $1\sigma$ errors that go into the figures and
tables. Rows are the matrices and columns the parameters. A blank means the
parameter is not in that model.

The bins of the binned-$\mathcal{P}(k)$ forecasts are shown separately, in
the second table. Together they are wide enough that pandas hides the
middle columns behind an ellipsis.

In [19]:
errors_table = pd.DataFrame(forecast_errors).T
cosmo_errors, bin_errors = split_bins(errors_table)

display(cosmo_errors)
bin_errors

,H0,logA,mnu,nnu,nrun,ns,ombh2,omch2,tau,A_ksz,HMCode_logT_AGN,n_ksz,w,wa
hd_camb_lensed_9param,0.317110,0.008473,0.028232,0.015340,0.001288,0.001932,0.000026,0.000389,0.004538,NaN,NaN,NaN,NaN,NaN
hd_camb_delensed_9param,0.315197,0.008397,0.028237,0.014311,0.001240,0.001809,0.000026,0.000366,0.004474,NaN,NaN,NaN,NaN,NaN
hd_camb_lensed_8param,0.311949,0.008098,0.026570,0.015126,NaN,0.001540,0.000026,0.000389,0.004351,NaN,NaN,NaN,NaN,NaN
hd_class_lensed_9param,0.310750,0.008522,0.028138,0.015312,0.001231,0.001935,0.000026,0.000392,0.004562,NaN,NaN,NaN,NaN,NaN
hd_class_lensed_8param,0.308474,0.008172,0.026839,0.014976,NaN,0.001507,0.000026,0.000391,0.004383,NaN,NaN,NaN,NaN,NaN
so_class_lensed_9param,0.406025,0.008865,0.031088,0.055575,0.003937,0.002952,0.000048,0.000866,0.004649,NaN,NaN,NaN,NaN,NaN
hd_camb_delensed_12param_feedback,0.303617,0.008551,0.029029,0.016262,0.001638,0.001835,0.000026,0.000378,0.004537,0.001396,0.004672,0.000864,NaN,NaN
hd_camb_delensed_9param_from_feedback,0.300852,0.008517,0.028938,0.014191,0.001145,0.001815,0.000026,0.000363,0.004514,NaN,NaN,NaN,NaN,NaN
hd_w0wa_lensed,1.211665,0.007521,NaN,NaN,0.001490,0.001728,0.000018,0.000360,0.004200,NaN,NaN,NaN,0.106895,0.263330
so_w0wa_lensed,1.603000,0.007783,NaN,NaN,0.002596,0.001860,0.000036,0.000392,0.004333,NaN,NaN,NaN,0.132840,0.307789


,Pk1,Pk2,Pk3,Pk4,Pk5,Pk6,Pk7,Pk8,Pk9,Pk10,Pk11
hd_binned_pk_lensed_11bins,4.941198e-11,2.366983e-11,1.353604e-11,1.496874e-11,1.584912e-11,1.642871e-11,2.668455e-11,2.957829e-11,1.557828e-10,1.020800e-09,7.214766e-09
so_binned_pk_lensed_7bins,4.969123e-11,2.415378e-11,1.434324e-11,1.604746e-11,1.664882e-11,2.823574e-11,1.738276e-10,NaN,NaN,NaN,NaN
hd_binned_pk_feedback_lensed,4.870898e-11,2.260140e-11,1.230983e-11,1.303783e-11,1.372417e-11,1.425024e-11,2.394005e-11,3.460799e-11,2.210891e-10,1.362901e-09,6.294956e-09
hd_binned_pk_feedback_pol_lensed,4.875680e-11,2.266344e-11,1.238923e-11,1.317611e-11,1.383770e-11,1.432051e-11,2.985847e-11,4.538825e-11,3.446872e-10,4.374707e-09,2.648918e-08


## Comparing against the saved forecasts

The same twenty matrices ship in `hdinitpk/data/fisher_matrices`, which is
what the plotting notebook reads. The ratio of the errors just computed to
the saved ones should be 1 everywhere. It is split the same way, with the
binned-$\mathcal{P}(k)$ bins in the second table.

In [20]:
def saved_errors(name):
    """1-sigma errors of a shipped matrix, keyed by CAMB names. The files
    built from a CLASS configuration keep CLASS names, so they are
    translated here too, to match `forecast_errors` above."""
    path = os.path.join(saved_matrix_dir, f'{name}.txt')
    if not os.path.isfile(path):
        return None
    return from_class_errors(
        fisher.get_fisher_errors(*fisher.load_fisher_matrix(path)))


ratios = {}
for name in forecast_errors:
    saved = saved_errors(name)
    if saved is None:
        continue
    new = forecast_errors[name]
    ratios[name] = {p: new[p] / saved[p] for p in new
                    if p in saved and saved[p] != 0}

comparison = pd.DataFrame(ratios).T
cosmo_ratios, bin_ratios = split_bins(comparison)

print(f'{len(ratios)} matrices compared against {saved_matrix_dir}')
display(cosmo_ratios)
bin_ratios

20 matrices compared against /gpfs/projects/SehgalGroup/zcheslog/cmb_hd/hdInitPk_test/hdinitpk/data/fisher_matrices


,H0,logA,mnu,nnu,nrun,ns,ombh2,omch2,tau,A_ksz,HMCode_logT_AGN,n_ksz,w,wa
hd_camb_lensed_9param,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
hd_camb_delensed_9param,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
hd_camb_lensed_8param,1.000000,1.000000,1.0,1.0,NaN,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
hd_class_lensed_9param,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
hd_class_lensed_8param,1.000000,1.000000,1.0,1.0,NaN,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
so_class_lensed_9param,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
hd_camb_delensed_12param_feedback,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.0,1.0,NaN,NaN
hd_camb_delensed_9param_from_feedback,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
hd_w0wa_lensed,1.000000,1.000000,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,1.000000,1.000000
so_w0wa_lensed,1.000000,1.000013,NaN,NaN,1.000714,0.999982,1.000187,1.000012,1.000016,NaN,NaN,NaN,0.999998,0.999991


,Pk1,Pk2,Pk3,Pk4,Pk5,Pk6,Pk7,Pk8,Pk9,Pk10,Pk11
hd_binned_pk_lensed_11bins,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.0,1.0,1.0,1.0
so_binned_pk_lensed_7bins,1.000021,1.000091,1.00007,1.000065,0.999869,0.999479,1.000007,NaN,NaN,NaN,NaN
hd_binned_pk_feedback_lensed,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.0,1.0,1.0,1.0
hd_binned_pk_feedback_pol_lensed,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.0,1.0,1.0,1.0


Once the ratios read 1, these can replace the saved set. The file names
already match what `hdInitPk_plots.ipynb` expects, so nothing else changes.

The matrices are saved with whatever names the configuration that built
them uses, so the CLASS ones keep CLASS names. That is the convention the
shipped set follows and what the plotting notebook expects, so the
translation above is for reading the tables only and is not applied here.

In [22]:
os.makedirs(matrix_out_dir, exist_ok=True)

for name, config, extra, call in matrices:
    if name not in forecast_errors:
        continue
    job = jobs[config]
    kwargs = dict(job['kwargs'], overwrite=False)
    kwargs.setdefault('hd_data_version', hd_data_version)
    kwargs.update(extra)
    fisherlib = hdinitPkfisher.Fisher(os.path.join(deriv_dir, job['dirname']), **kwargs)
    matrix, matrix_params = fisherlib.get_fisher(use_H0=use_H0, save=False,
                                                 **call)
    fisher.save_fisher_matrix(os.path.join(matrix_out_dir, f'{name}.txt'),
                              matrix, matrix_params)

print(f'{len(forecast_errors)} matrices written to {matrix_out_dir}')

/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hdfisher/fisher.py:956: UserWarning: Removing 'theta' from the list of varied parameters, because `use_H0=True`.
  warnings.warn(msg)
/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hdfisher/fisher.py:884: UserWarning: `use_class=True`: delensed spectra are not calculated with CLASS. No delensed derivatives will be calculated, and a Fisher matrix cannot be calculated from delensed spectra. Use `cmb_type='lensed'`, or set `use_class=False` to use CAMB.
  warnings.warn(msg)
/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hdfisher/fisher.py:807: UserWarning: Ignoring the `hd_lmax` and `include_fg` arguments for `exp = 'so'`.
  warnings.warn(msg1)
/gpfs/home/zcheslog/.local/lib/python3.11/site-packages/hdfisher/fisher.py:809: UserWarning: NOTE that we only have mock covariance matrices for *delensed* (as opposed to lensed) CMB spectra for `exp = 'so'` (i.e., we can only calculate a Fisher matrix from delensed power spe

20 matrices written to /gpfs/projects/SehgalGroup/zcheslog/cmb_hd/hdInitPk_test/hdinitpk/data/user_generated_data/fisher_matrices


## Forecasting something else

To add a model the paper does not cover you need a fiducial-parameter file
and a step-size file in `hdinitpk/data/fisher_fid_params` and
`hdinitpk/data/fisher_steps` (every parameter in the step file needs a
fiducial value), then a new entry in `jobs`, a derivative run, and a new
entry in `matrices`.

For a binned $\mathcal{P}(k)$ forecast pass `binned_pk=True`, `bin_edges`,
and a `pk_frac_step` that matches the step sizes of the bin amplitudes. The parameter file then needs
`nkbins`, the bin centers `k1` to `kN`, the amplitudes `Pk1` to `PkN` (or
`eneg2tauPk1` and so on), and an `effective_ns_for_nonlinear` entry.
`example_calc_binnedPk_forecasts.py` in the repository root is one complete
binned forecast, with the derivatives, the matrix and the errors annotated
line by line, if it is easier to read as a script.